# Sarcasm Detection — Unified Pipeline

This single notebook runs the complete sarcasm detection pipeline end-to-end:

1. **Data preparation** — load, clean, and split the dataset (70 / 15 / 15)
2. **TF-IDF + Logistic Regression** — fast baseline trained on the full dataset
3. **Context-free DeBERTa-v3-base + LoRA** — transformer on reply text only (full dataset)
4. **Context-aware DeBERTa-v3-base + LoRA** — primary model on parent+reply pairs (full dataset)
5. **Retrieval-augmented ensemble** — BGE embeddings + FAISS + validation-calibrated stacker
6. **LLM prompt artifacts** — zero-shot and few-shot prompt templates

Dataset: `danofer/sarcasm` (Reddit, ~1 M examples)  
Primary model: `microsoft/deberta-v3-base` fine-tuned with PEFT LoRA (r=16, alpha=32, dropout=0.05)  
Target modules: `query_proj`, `key_proj`, `value_proj` + saved `classifier` and `pooler`

> **GPU required** for transformer training. On a Kaggle T4 the full pipeline takes ~3–4 h.


## Install dependencies


In [ ]:
!pip install -q transformers accelerate peft datasets evaluate sentencepiece safetensors pyarrow joblib sentence-transformers faiss-cpu


## Shared utility definitions

All helper classes and functions are defined inline — no external module file needed.


In [ ]:
"""Shared Kaggle utilities for the sarcasm detection notebooks.

The notebooks import this module instead of duplicating large helper cells.
Functions are written to be usable in Kaggle CPU/GPU sessions and to degrade
cleanly when optional packages such as pyarrow, faiss, or sentence-transformers
are unavailable.
"""

from __future__ import annotations

import gc
import inspect
import json
import os
import random
import re
import shutil
import sys
import time
import warnings
import zipfile
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd


warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")


@dataclass
class SarcasmConfig:
    dataset_slug: str = "danofer/sarcasm"
    input_dir: str = "/kaggle/input/"
    working_dir: str = "/kaggle/working/"
    output_dir: str = "/kaggle/working/"
    kaggle_notebook_input_names: list[str] = field(
        default_factory=lambda: [
            "01-data-and-tfidf-baseline",
            "02-context-free-transformer",
            "03-context-aware-transformer",
            "04-retrieval-and-llm-prompts",
            "01_data_and_tfidf_baseline",
            "02_context_free_transformer",
            "03_context_aware_transformer",
            "04_retrieval_and_llm_prompts",
        ]
    )
    model_name: str = "microsoft/deberta-v3-base"
    optional_large_model_name: str = "microsoft/deberta-v3-large"
    random_seed: int = 42
    debug: bool = False
    sample_size: int = 20000
    max_length: int = 256
    per_device_train_batch_size: int = 8
    per_device_eval_batch_size: int = 16
    gradient_accumulation_steps: int = 4
    epochs: int = 2
    learning_rate: float = 2e-4
    weight_decay: float = 0.01
    warmup_ratio: float = 0.06
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    lora_target_modules: list[str] = field(
        default_factory=lambda: ["query_proj", "key_proj", "value_proj"]
    )
    lora_modules_to_save: list[str] = field(default_factory=lambda: ["classifier", "pooler"])
    eval_steps: int = 3000
    save_steps: int = 3000
    logging_steps: int = 200
    save_total_limit: int = 2
    max_transformer_train_rows: int | None = None
    max_transformer_eval_rows: int | None = None
    cpu_max_transformer_train_rows: int = 2000
    cpu_max_transformer_eval_rows: int = 500
    require_gpu_for_full_transformer: bool = True
    resume_from_checkpoint: bool = True
    sliding_window_stride: int = 64
    max_windows_per_example: int = 4
    protected_min_context_tokens: int = 32
    run_tfidf: bool = True
    run_context_free_transformer: bool = True
    run_context_aware_transformer: bool = True
    run_rag: bool = True
    run_llm_prompts: bool = True
    retrieval_embedding_model: str = "BAAI/bge-base-en-v1.5"
    retrieval_batch_size: int = 64
    retrieval_k: int = 5
    max_retrieval_train_rows: int = 50000
    max_retrieval_eval_rows: int | None = None
    max_llm_prompt_rows: int = 100

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)


CONFIG = SarcasmConfig()

COMPARISON_COLUMNS = [
    "model_name",
    "input_type",
    "threshold",
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "f1",
    "macro_f1",
    "weighted_f1",
    "roc_auc",
    "average_precision",
    "test_samples",
    "positive_samples",
    "negative_samples",
    "notes",
]

PLOT_METRICS = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "f1",
    "macro_f1",
    "weighted_f1",
    "roc_auc",
    "average_precision",
]


def cfg(**overrides: Any) -> SarcasmConfig:
    data = CONFIG.to_dict()
    data.update(overrides)
    return SarcasmConfig(**data)


def output_path(config: SarcasmConfig = CONFIG) -> Path:
    path = Path(config.output_dir)
    path.mkdir(parents=True, exist_ok=True)
    return path


def make_jsonable(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(key): make_jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [make_jsonable(item) for item in value]
    if isinstance(value, np.ndarray):
        return make_jsonable(value.tolist())
    if isinstance(value, np.generic):
        return make_jsonable(value.item())
    if isinstance(value, float):
        if np.isnan(value) or np.isinf(value):
            return None
        return value
    return value


def write_json(path: str | Path, data: Any) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(make_jsonable(data), indent=2), encoding="utf-8")
    return path


def set_all_seeds(seed: int = 42, deterministic: bool = False) -> None:
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        from transformers import set_seed

        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        set_seed(seed)
        if deterministic:
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
    except Exception:
        pass


def version_log(config: SarcasmConfig = CONFIG) -> dict[str, Any]:
    packages = {}
    for name in [
        "python",
        "numpy",
        "pandas",
        "sklearn",
        "torch",
        "transformers",
        "peft",
        "datasets",
        "sentence_transformers",
        "faiss",
    ]:
        try:
            if name == "python":
                packages[name] = sys.version
            elif name == "sklearn":
                import sklearn

                packages[name] = sklearn.__version__
            elif name == "faiss":
                import faiss

                packages[name] = getattr(faiss, "__version__", "available")
            else:
                module = __import__(name)
                packages[name] = getattr(module, "__version__", "available")
        except Exception as exc:
            packages[name] = f"not available: {exc.__class__.__name__}"

    gpu_info = []
    try:
        import torch

        for index in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(index)
            gpu_info.append(
                {
                    "index": index,
                    "name": torch.cuda.get_device_name(index),
                    "total_memory_gb": round(props.total_memory / (1024**3), 2),
                }
            )
    except Exception:
        pass

    return {
        "packages": packages,
        "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
        "gpus": gpu_info,
        "config": config.to_dict(),
    }


def save_run_metadata(config: SarcasmConfig = CONFIG) -> None:
    out = output_path(config)
    write_json(out / "run_config.json", config.to_dict())
    write_json(out / "version_log.json", version_log(config))


def cleanup_memory() -> None:
    gc.collect()
    try:
        import torch

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass


def kaggle_notebook_input_roots(config: SarcasmConfig = CONFIG) -> list[Path]:
    input_root = Path(config.input_dir)
    roots: list[Path] = []
    if not input_root.exists():
        return roots

    for notebook_name in config.kaggle_notebook_input_names:
        direct_root = input_root / notebook_name
        if direct_root.exists() and direct_root.is_dir():
            roots.append(direct_root)
        try:
            roots.extend([p for p in input_root.rglob(notebook_name) if p.is_dir()])
        except Exception:
            pass

    unique_roots = []
    seen = set()
    for root in roots:
        resolved = root.resolve()
        if resolved not in seen:
            seen.add(resolved)
            unique_roots.append(root)
    return unique_roots


def search_roots(config: SarcasmConfig = CONFIG) -> list[Path]:
    roots = [Path(config.working_dir), Path.cwd()]
    roots.extend(kaggle_notebook_input_roots(config))
    roots.append(Path(config.input_dir))
    unique_roots = []
    seen = set()
    for root in roots:
        resolved = root.resolve() if root.exists() else root
        if resolved not in seen:
            seen.add(resolved)
            unique_roots.append(root)
    return unique_roots


def prepare_input_search(config: SarcasmConfig = CONFIG) -> None:
    print("Input search order:")
    for root in search_roots(config):
        print(f"- {root} [{'exists' if root.exists() else 'missing'}]")


def find_file_recursive(filename: str, config: SarcasmConfig = CONFIG) -> Path | None:
    matches: list[Path] = []
    roots = search_roots(config)
    for root in roots:
        if root.exists():
            matches.extend(sorted(root.rglob(filename)))
    if not matches:
        return None

    def rank(path: Path) -> tuple[int, int]:
        resolved = path.resolve()
        for index, root in enumerate(roots):
            if not root.exists():
                continue
            root_resolved = root.resolve()
            if resolved == root_resolved or root_resolved in resolved.parents:
                return (index, len(str(resolved)))
        return (len(roots), len(str(resolved)))

    return sorted(set(matches), key=rank)[0]


def require_file(filename: str, message: str, config: SarcasmConfig = CONFIG) -> Path:
    path = find_file_recursive(filename, config)
    if path is None:
        print(message)
        prepare_input_search(config)
        raise FileNotFoundError(message)
    print(f"Found {filename}: {path}")
    return path


def copy_to_working(path: str | Path, config: SarcasmConfig = CONFIG) -> Path:
    source = Path(path)
    destination = output_path(config) / source.name
    if source.exists() and source.resolve() != destination.resolve():
        shutil.copy2(source, destination)
    return destination


def zip_outputs(zip_name: str, items: Iterable[str | Path], config: SarcasmConfig = CONFIG) -> Path:
    out = output_path(config)
    zip_path = out / zip_name
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
        for item in items:
            path = Path(item)
            if not path.is_absolute():
                path = out / path
            if path.is_file():
                zipf.write(path, arcname=path.name)
            elif path.is_dir():
                for file_path in path.rglob("*"):
                    if file_path.is_file():
                        zipf.write(file_path, arcname=str(file_path.relative_to(out)))
            else:
                print(f"Skipping missing item: {path}")
    print(f"Created ZIP: {zip_path}")
    return zip_path


def read_table(path: str | Path) -> pd.DataFrame:
    path = Path(path)
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, low_memory=False)


def write_table(frame: pd.DataFrame, stem: str, config: SarcasmConfig = CONFIG) -> dict[str, Path | None]:
    out = output_path(config)
    csv_path = out / f"{stem}.csv"
    parquet_path = out / f"{stem}.parquet"
    frame.to_csv(csv_path, index=False)
    try:
        frame.to_parquet(parquet_path, index=False)
    except Exception as exc:
        print(f"Could not write {parquet_path.name}: {exc}")
        parquet_path = None
    return {"csv": csv_path, "parquet": parquet_path}


def read_split(split_name: str, config: SarcasmConfig = CONFIG) -> pd.DataFrame:
    parquet = find_file_recursive(f"{split_name}_split.parquet", config)
    if parquet is not None:
        return pd.read_parquet(parquet)
    csv_path = require_file(
        f"{split_name}_split.csv",
        f"Missing {split_name}_split.csv/parquet. Run Notebook 1 first or attach its outputs.",
        config,
    )
    return pd.read_csv(csv_path, low_memory=False)


def find_csv_files(root_dir: str | Path) -> list[Path]:
    root = Path(root_dir)
    if not root.exists():
        return []
    return sorted(root.rglob("*.csv"))


def choose_dataset_csv(csv_files: list[Path]) -> Path:
    if not csv_files:
        raise FileNotFoundError(
            "No CSV files were found. Attach the Kaggle dataset danofer/sarcasm."
        )
    scored = []
    for path in csv_files:
        name = path.name.lower()
        keyword_score = sum(keyword in name for keyword in ["train", "sarcasm", "reddit"])
        scored.append(
            {
                "path": path,
                "name": path.name,
                "size_mb": path.stat().st_size / (1024 * 1024),
                "keyword_score": keyword_score,
            }
        )
    found_df = pd.DataFrame(scored).sort_values(
        ["keyword_score", "size_mb"], ascending=[False, False]
    )
    print("Candidate CSV files:")
    print(found_df[["name", "size_mb", "keyword_score"]].head(10))
    return Path(found_df.iloc[0]["path"])


def normalized_name(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(name).lower())


def find_column(df: pd.DataFrame, exact_name: str, candidates: list[str]) -> str | None:
    if exact_name in df.columns:
        return exact_name
    normalized_to_original = {normalized_name(column): column for column in df.columns}
    for candidate in candidates:
        normalized = normalized_name(candidate)
        if normalized in normalized_to_original:
            return normalized_to_original[normalized]
    return None


def to_clean_text(value: Any) -> str:
    if pd.isna(value):
        return ""
    text = str(value)
    text = text.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def convert_label(value: Any) -> int | None:
    if pd.isna(value):
        return None
    if isinstance(value, str):
        cleaned = value.strip().lower()
        if cleaned in {"1", "true", "yes", "sarcastic", "s"}:
            return 1
        if cleaned in {"0", "false", "no", "not sarcastic", "non-sarcastic", "n"}:
            return 0
    try:
        numeric = int(float(value))
    except Exception:
        return None
    if numeric in {0, 1}:
        return numeric
    return None


def load_and_clean_dataset(config: SarcasmConfig = CONFIG) -> pd.DataFrame:
    dataset_path = choose_dataset_csv(find_csv_files(config.input_dir))
    raw_df = pd.read_csv(dataset_path, low_memory=False)
    print("Raw dataset shape:", raw_df.shape)
    print("Dataset path:", dataset_path)
    print("Available columns:", list(raw_df.columns))

    target_text_candidates = ["comment", "reply", "target", "body", "text"]
    context_candidates = ["parent_comment", "parent", "context", "previous_comment"]
    label_candidates = ["label", "sarcasm", "is_sarcastic"]

    comment_col = find_column(raw_df, "comment", target_text_candidates)
    parent_col = find_column(raw_df, "parent_comment", context_candidates)
    label_col = find_column(raw_df, "label", label_candidates)
    subreddit_col = find_column(raw_df, "subreddit", ["subreddit", "sub", "community"])

    missing = []
    if comment_col is None:
        missing.append("target reply text column")
    if parent_col is None:
        missing.append("parent/context column")
    if label_col is None:
        missing.append("label column")
    if missing:
        raise ValueError("Missing required columns: " + ", ".join(missing))

    keep_columns = [comment_col, parent_col, label_col]
    if subreddit_col is not None:
        keep_columns.append(subreddit_col)
    df = raw_df[keep_columns].copy()
    rename_map = {comment_col: "comment", parent_col: "parent_comment", label_col: "label"}
    if subreddit_col is not None:
        rename_map[subreddit_col] = "subreddit"
    df = df.rename(columns=rename_map)

    df["comment"] = df["comment"].map(to_clean_text)
    df["parent_comment"] = df["parent_comment"].map(to_clean_text)
    df["label"] = df["label"].map(convert_label)
    before = len(df)
    df = df.dropna(subset=["label"])
    df["label"] = df["label"].astype(int)
    df = df[df["comment"].str.len() > 0]
    df = df[df["parent_comment"].str.len() > 0]
    df = df[df["label"].isin([0, 1])]
    df = df.drop_duplicates(subset=["parent_comment", "comment", "label"])
    df = df.reset_index(drop=True)
    df.insert(0, "example_id", np.arange(len(df)))
    if config.debug and len(df) > config.sample_size:
        from sklearn.model_selection import train_test_split

        df, _ = train_test_split(
            df,
            train_size=config.sample_size,
            stratify=df["label"],
            random_state=config.random_seed,
        )
        df = df.sort_values("example_id").reset_index(drop=True)
        print(f"DEBUG mode: using {len(df):,} rows.")
    print(f"Rows before cleaning: {before:,}")
    print(f"Rows after cleaning: {len(df):,}")
    return df


def create_splits(df: pd.DataFrame, config: SarcasmConfig = CONFIG) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    from sklearn.model_selection import train_test_split

    split_columns = ["example_id", "parent_comment", "comment", "label"]
    if "subreddit" in df.columns:
        split_columns.append("subreddit")
    split_df = df[split_columns].copy()
    train_df, temp_df = train_test_split(
        split_df,
        test_size=0.30,
        stratify=split_df["label"],
        random_state=config.random_seed,
    )
    validation_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        stratify=temp_df["label"],
        random_state=config.random_seed,
    )
    return (
        train_df.reset_index(drop=True),
        validation_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
    )


def save_split_artifacts(
    train_df: pd.DataFrame,
    validation_df: pd.DataFrame,
    test_df: pd.DataFrame,
    config: SarcasmConfig = CONFIG,
) -> None:
    write_table(train_df, "train_split", config)
    write_table(validation_df, "validation_split", config)
    write_table(test_df, "test_split", config)
    print("Train shape:", train_df.shape)
    print("Validation shape:", validation_df.shape)
    print("Test shape:", test_df.shape)


def _word_count_series(series: pd.Series) -> pd.Series:
    return series.fillna("").astype(str).str.split().str.len()


def length_diagnostics(df: pd.DataFrame) -> dict[str, Any]:
    comment_words = _word_count_series(df["comment"])
    parent_words = _word_count_series(df["parent_comment"])
    pair_words = comment_words + parent_words
    comment_chars = df["comment"].fillna("").astype(str).str.len()
    pair_chars = comment_chars + df["parent_comment"].fillna("").astype(str).str.len()

    def describe(series: pd.Series) -> dict[str, Any]:
        return {
            "mean": float(series.mean()),
            "p50": float(series.quantile(0.50)),
            "p90": float(series.quantile(0.90)),
            "p95": float(series.quantile(0.95)),
            "p99": float(series.quantile(0.99)),
            "max": int(series.max()),
        }

    thresholds = [64, 96, 128, 160, 192, 224, 256, 320, 384, 512]
    return {
        "rows": int(len(df)),
        "comment_words": describe(comment_words),
        "parent_plus_comment_words": describe(pair_words),
        "comment_chars": describe(comment_chars),
        "parent_plus_comment_chars": describe(pair_chars),
        "rows_above_pair_word_threshold": {
            str(threshold): {
                "count": int((pair_words > threshold).sum()),
                "pct": float((pair_words > threshold).mean() * 100),
            }
            for threshold in thresholds
        },
    }


def save_length_diagnostics(
    train_df: pd.DataFrame,
    validation_df: pd.DataFrame,
    test_df: pd.DataFrame,
    config: SarcasmConfig = CONFIG,
) -> dict[str, Any]:
    diagnostics = {
        "train": length_diagnostics(train_df),
        "validation": length_diagnostics(validation_df),
        "test": length_diagnostics(test_df),
    }
    write_json(output_path(config) / "length_diagnostics.json", diagnostics)
    return diagnostics


def softmax(logits: np.ndarray) -> np.ndarray:
    logits = np.asarray(logits)
    logits = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(logits)
    return exp / exp.sum(axis=1, keepdims=True)


def compute_binary_metrics(
    y_true: Iterable[int],
    y_pred: Iterable[int],
    y_prob: Iterable[float] | None,
    model_name: str,
    input_type: str,
    notes: str,
    threshold: float | None = None,
) -> dict[str, Any]:
    from sklearn.metrics import (
        accuracy_score,
        average_precision_score,
        balanced_accuracy_score,
        classification_report,
        confusion_matrix,
        f1_score,
        precision_score,
        recall_score,
        roc_auc_score,
    )

    y_true = np.asarray(list(y_true)).astype(int)
    y_pred = np.asarray(list(y_pred)).astype(int)
    y_prob_array = None if y_prob is None else np.asarray(list(y_prob)).astype(float)
    record = {
        "model_name": model_name,
        "input_type": input_type,
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "roc_auc": None,
        "average_precision": None,
        "test_samples": int(len(y_true)),
        "positive_samples": int((y_true == 1).sum()),
        "negative_samples": int((y_true == 0).sum()),
        "notes": notes,
        "classification_report": classification_report(
            y_true, y_pred, output_dict=True, zero_division=0
        ),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
    }
    if y_prob_array is not None and len(np.unique(y_true)) == 2:
        record["roc_auc"] = roc_auc_score(y_true, y_prob_array)
        record["average_precision"] = average_precision_score(y_true, y_prob_array)
    return make_jsonable(record)


def save_confusion_matrix(
    y_true: Iterable[int],
    y_pred: Iterable[int],
    title: str,
    filename: str,
    config: SarcasmConfig = CONFIG,
) -> None:
    import matplotlib.pyplot as plt
    import seaborn as sns
    from sklearn.metrics import confusion_matrix

    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5.2, 4.4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap=sns.light_palette("#0072B2", as_cmap=True),
        cbar=False,
        linewidths=0.5,
        linecolor="white",
        xticklabels=["Non-sarcastic", "Sarcastic"],
        yticklabels=["Non-sarcastic", "Sarcastic"],
        ax=ax,
    )
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title(title)
    fig.tight_layout()
    path = output_path(config) / filename
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.savefig(path.with_suffix(".pdf"), bbox_inches="tight")
    plt.close(fig)


def save_metrics_record(record: dict[str, Any], filename: str, config: SarcasmConfig = CONFIG) -> None:
    write_json(output_path(config) / filename, record)
    print(f"Saved {filename}")
    for key in PLOT_METRICS:
        value = record.get(key)
        print(f"{key}: {value if value is None else float(value):.4f}" if value is not None else f"{key}: NA")


def load_metric_records(config: SarcasmConfig = CONFIG) -> list[dict[str, Any]]:
    records = []
    for pattern in ["*_metrics.json", "retrieval_metrics.json", "llm_metrics.json"]:
        for path in sorted(set(p for root in search_roots(config) if root.exists() for p in root.rglob(pattern))):
            if path.name in {"retrieval_knn_diagnostic_metrics.json"}:
                continue
            try:
                record = json.loads(path.read_text(encoding="utf-8"))
                if isinstance(record, list):
                    records.extend(record)
                elif "model_name" in record:
                    records.append(record)
            except Exception:
                continue
    seen = set()
    unique = []
    for record in records:
        key = (record.get("model_name"), record.get("input_type"))
        if key in seen:
            continue
        seen.add(key)
        unique.append(record)
    return unique


def update_comparison_outputs(config: SarcasmConfig = CONFIG) -> pd.DataFrame:
    records = load_metric_records(config)
    rows = []
    for record in records:
        rows.append({column: record.get(column, np.nan) for column in COMPARISON_COLUMNS})
    metrics_df = pd.DataFrame(rows, columns=COMPARISON_COLUMNS)
    metrics_df.to_csv(output_path(config) / "metrics_comparison.csv", index=False)
    print("Saved metrics_comparison.csv")
    return metrics_df


def best_threshold_for_metric(
    scores: Iterable[float],
    labels: Iterable[int],
    metric: str = "f1",
    lower: float = 0.05,
    upper: float = 0.95,
    steps: int = 181,
) -> float:
    from sklearn.metrics import accuracy_score, f1_score

    scores = np.asarray(list(scores)).astype(float)
    labels = np.asarray(list(labels)).astype(int)
    best_value = -1.0
    best_threshold = 0.5
    for threshold in np.linspace(lower, upper, steps):
        predictions = (scores >= threshold).astype(int)
        if metric == "accuracy":
            value = accuracy_score(labels, predictions)
        else:
            value = f1_score(labels, predictions, zero_division=0)
        if value > best_value:
            best_value = value
            best_threshold = float(threshold)
    return best_threshold


def stratified_limit_frame(
    frame: pd.DataFrame,
    max_rows: int | None,
    name: str,
    config: SarcasmConfig = CONFIG,
) -> pd.DataFrame:
    if max_rows is None or len(frame) <= max_rows:
        return frame.reset_index(drop=True)
    from sklearn.model_selection import train_test_split

    if "label" in frame.columns and frame["label"].nunique() > 1:
        sampled, _ = train_test_split(
            frame,
            train_size=max_rows,
            stratify=frame["label"],
            random_state=config.random_seed,
        )
    else:
        sampled = frame.sample(n=max_rows, random_state=config.random_seed)
    sampled = sampled.sort_values("example_id") if "example_id" in sampled.columns else sampled
    print(f"Using {len(sampled):,} rows for {name}.")
    return sampled.reset_index(drop=True)


def prepare_transformer_runtime_frames(
    train_df: pd.DataFrame,
    validation_df: pd.DataFrame,
    test_df: pd.DataFrame,
    config: SarcasmConfig = CONFIG,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    try:
        import torch
    except Exception:
        torch = None
    has_gpu = bool(torch is not None and torch.cuda.is_available())
    if has_gpu:
        train_limit = config.max_transformer_train_rows
        eval_limit = config.max_transformer_eval_rows
        print("Using full Kaggle GPU transformer configuration.")
    else:
        if config.require_gpu_for_full_transformer and not config.debug:
            raise RuntimeError(
                "Full transformer training requires a Kaggle GPU. Enable GPU or set DEBUG=True."
            )
        train_limit = config.cpu_max_transformer_train_rows
        eval_limit = config.cpu_max_transformer_eval_rows
        print("CPU/debug transformer run: limiting rows and using lighter settings.")
    return (
        stratified_limit_frame(train_df, train_limit, "train", config),
        stratified_limit_frame(validation_df, eval_limit, "validation", config),
        stratified_limit_frame(test_df, eval_limit, "test", config),
    )


class LazySarcasmDataset:
    def __init__(
        self,
        dataframe: pd.DataFrame,
        tokenizer: Any,
        input_type: str,
        config: SarcasmConfig = CONFIG,
    ) -> None:
        import torch

        self.torch = torch
        self.labels = dataframe["label"].astype(int).tolist()
        self.comments = dataframe["comment"].fillna("").astype(str).tolist()
        self.parents = dataframe["parent_comment"].fillna("").astype(str).tolist()
        self.tokenizer = tokenizer
        self.input_type = input_type
        self.max_length = config.max_length

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> dict[str, Any]:
        if self.input_type == "context_free":
            encoded = self.tokenizer(
                self.comments[idx],
                truncation=True,
                max_length=self.max_length,
            )
        elif self.input_type == "context_aware":
            try:
                encoded = self.tokenizer(
                    text=self.parents[idx],
                    text_pair=self.comments[idx],
                    truncation="only_first",
                    max_length=self.max_length,
                )
            except Exception:
                encoded = self.tokenizer(
                    text=self.parents[idx],
                    text_pair=self.comments[idx],
                    truncation=True,
                    max_length=self.max_length,
                )
        else:
            raise ValueError("input_type must be context_free or context_aware")
        encoded["labels"] = self.labels[idx]
        return encoded


def pair_special_token_count(tokenizer: Any) -> int:
    pair_ids = tokenizer("", "", add_special_tokens=True)["input_ids"]
    return len(pair_ids)


def example_needs_windowing(
    tokenizer: Any,
    parent: str,
    comment: str,
    max_length: int,
) -> bool:
    parent_ids = tokenizer(parent, add_special_tokens=False)["input_ids"]
    comment_ids = tokenizer(comment, add_special_tokens=False)["input_ids"]
    return len(parent_ids) + len(comment_ids) + pair_special_token_count(tokenizer) > max_length


def encode_context_aware_windows(
    tokenizer: Any,
    parent: str,
    comment: str,
    config: SarcasmConfig = CONFIG,
) -> list[dict[str, Any]]:
    parent_ids = tokenizer(parent, add_special_tokens=False)["input_ids"]
    comment_ids = tokenizer(comment, add_special_tokens=False)["input_ids"]
    special_tokens = pair_special_token_count(tokenizer)
    max_comment_len = max(8, config.max_length - special_tokens - 1)
    if len(comment_ids) > max_comment_len:
        comment_ids = comment_ids[:max_comment_len]
    parent_window = config.max_length - len(comment_ids) - special_tokens
    if parent_window <= 0:
        encoded = tokenizer.prepare_for_model(
            [],
            comment_ids,
            add_special_tokens=True,
            max_length=config.max_length,
            truncation=True,
        )
        return [encoded]
    if len(parent_ids) + len(comment_ids) + special_tokens <= config.max_length:
        encoded = tokenizer.prepare_for_model(
            parent_ids,
            comment_ids,
            add_special_tokens=True,
            max_length=config.max_length,
            truncation=True,
        )
        return [encoded]

    step = max(1, parent_window - config.sliding_window_stride)
    windows = []
    for start in range(0, len(parent_ids), step):
        windows.append(parent_ids[start : start + parent_window])
        if start + parent_window >= len(parent_ids):
            break
    if len(windows) > config.max_windows_per_example:
        selected = [windows[0]]
        selected.extend(windows[-(config.max_windows_per_example - 1) :])
        windows = selected

    return [
        tokenizer.prepare_for_model(
            window,
            comment_ids,
            add_special_tokens=True,
            max_length=config.max_length,
            truncation=True,
        )
        for window in windows
    ]


def encode_single_example(
    tokenizer: Any,
    parent: str,
    comment: str,
    input_type: str,
    config: SarcasmConfig = CONFIG,
    use_windows: bool = False,
) -> list[dict[str, Any]]:
    if input_type == "context_free":
        return [
            tokenizer(
                comment,
                truncation=True,
                max_length=config.max_length,
            )
        ]
    if input_type != "context_aware":
        raise ValueError("input_type must be context_free or context_aware")
    if use_windows:
        return encode_context_aware_windows(tokenizer, parent, comment, config)
    try:
        return [
            tokenizer(
                text=parent,
                text_pair=comment,
                truncation="only_first",
                max_length=config.max_length,
            )
        ]
    except Exception:
        return [
            tokenizer(
                text=parent,
                text_pair=comment,
                truncation=True,
                max_length=config.max_length,
            )
        ]


def sliding_window_diagnostics(
    dataframe: pd.DataFrame,
    tokenizer: Any,
    config: SarcasmConfig = CONFIG,
) -> dict[str, Any]:
    overflow = 0
    windows = []
    for _, row in dataframe.iterrows():
        encoded = encode_context_aware_windows(
            tokenizer,
            str(row["parent_comment"]),
            str(row["comment"]),
            config,
        )
        count = len(encoded)
        windows.append(count)
        if count > 1:
            overflow += 1
    return {
        "rows": int(len(dataframe)),
        "overflow_rows": int(overflow),
        "overflow_pct": float(100 * overflow / max(1, len(dataframe))),
        "mean_windows": float(np.mean(windows)) if windows else 0.0,
        "max_windows": int(max(windows)) if windows else 0,
        "max_windows_per_example": config.max_windows_per_example,
        "stride": config.sliding_window_stride,
    }


def predict_logits_dataframe(
    model: Any,
    tokenizer: Any,
    dataframe: pd.DataFrame,
    input_type: str,
    config: SarcasmConfig = CONFIG,
    use_windows: bool = False,
) -> np.ndarray:
    import torch
    from transformers import DataCollatorWithPadding

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()
    collator = DataCollatorWithPadding(
        tokenizer=tokenizer,
        pad_to_multiple_of=8 if torch.cuda.is_available() else None,
    )
    rows_for_windows = []
    encoded_windows = []
    for row_index, row in dataframe.reset_index(drop=True).iterrows():
        encodings = encode_single_example(
            tokenizer,
            str(row["parent_comment"]),
            str(row["comment"]),
            input_type,
            config,
            use_windows=use_windows,
        )
        encoded_windows.extend(encodings)
        rows_for_windows.extend([row_index] * len(encodings))

    all_logits = []
    batch_size = config.per_device_eval_batch_size
    with torch.no_grad():
        for start in range(0, len(encoded_windows), batch_size):
            batch = encoded_windows[start : start + batch_size]
            model_inputs = collator(batch)
            model_inputs = {key: value.to(device) for key, value in model_inputs.items()}
            outputs = model(**model_inputs)
            all_logits.append(outputs.logits.detach().cpu().numpy())
    window_logits = np.vstack(all_logits)
    row_logits = np.zeros((len(dataframe), window_logits.shape[1]), dtype=np.float32)
    row_counts = np.zeros(len(dataframe), dtype=np.float32)
    for row_index, logits in zip(rows_for_windows, window_logits):
        row_logits[row_index] += logits
        row_counts[row_index] += 1
    row_logits = row_logits / np.maximum(row_counts[:, None], 1.0)
    return row_logits


def _training_args(output_dir: Path, run_name: str, config: SarcasmConfig) -> Any:
    import torch
    from transformers import TrainingArguments

    signature = inspect.signature(TrainingArguments.__init__)
    kwargs = {
        "output_dir": str(output_dir),
        "learning_rate": config.learning_rate,
        "num_train_epochs": config.epochs,
        "per_device_train_batch_size": config.per_device_train_batch_size,
        "per_device_eval_batch_size": config.per_device_eval_batch_size,
        "gradient_accumulation_steps": config.gradient_accumulation_steps,
        "fp16": torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
        "bf16": torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
        "gradient_checkpointing": True,
        "group_by_length": True,
        "save_total_limit": config.save_total_limit,
        "load_best_model_at_end": True,
        "metric_for_best_model": "f1",
        "greater_is_better": True,
        "warmup_ratio": config.warmup_ratio,
        "weight_decay": config.weight_decay,
        "max_grad_norm": 1.0,
        "logging_steps": config.logging_steps,
        "report_to": [],
        "seed": config.random_seed,
        "dataloader_num_workers": 2,
        "remove_unused_columns": False,
    }
    strategy_key = "eval_strategy" if "eval_strategy" in signature.parameters else "evaluation_strategy"
    kwargs[strategy_key] = "steps"
    kwargs["save_strategy"] = "steps"
    kwargs["eval_steps"] = config.eval_steps
    kwargs["save_steps"] = config.save_steps
    if "run_name" in signature.parameters:
        kwargs["run_name"] = run_name
    if "optim" in signature.parameters:
        kwargs["optim"] = "adamw_torch"
    return TrainingArguments(**kwargs)


def trainer_compute_metrics(eval_pred: Any) -> dict[str, float]:
    logits, labels = eval_pred
    probs = softmax(logits)[:, 1]
    preds = (probs >= 0.5).astype(int)
    record = compute_binary_metrics(
        y_true=labels,
        y_pred=preds,
        y_prob=probs,
        model_name="validation",
        input_type="validation",
        notes="Trainer validation metrics at threshold 0.5.",
        threshold=0.5,
    )
    return {
        "accuracy": record["accuracy"],
        "balanced_accuracy": record["balanced_accuracy"],
        "precision": record["precision"],
        "recall": record["recall"],
        "f1": record["f1"],
        "macro_f1": record["macro_f1"],
        "weighted_f1": record["weighted_f1"],
        "roc_auc": record["roc_auc"] if record["roc_auc"] is not None else 0.0,
        "average_precision": record["average_precision"]
        if record["average_precision"] is not None
        else 0.0,
    }


def train_deberta_lora_classifier(
    train_df: pd.DataFrame,
    validation_df: pd.DataFrame,
    test_df: pd.DataFrame,
    input_type: str,
    run_label: str,
    adapter_dir_name: str,
    predictions_prefix: str,
    metrics_filename: str,
    config: SarcasmConfig = CONFIG,
    use_windows_for_final_prediction: bool = False,
) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    import torch
    from peft import LoraConfig, TaskType, get_peft_model
    from transformers import (
        AutoModelForSequenceClassification,
        AutoTokenizer,
        DataCollatorWithPadding,
        EarlyStoppingCallback,
        Trainer,
    )
    from transformers.trainer_utils import get_last_checkpoint

    cleanup_memory()
    set_all_seeds(config.random_seed)
    out = output_path(config)
    adapter_dir = out / adapter_dir_name
    trainer_output_dir = out / f"{adapter_dir_name}_trainer"
    trainer_output_dir.mkdir(parents=True, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=True)
    torch_dtype = torch.float16 if torch.cuda.is_available() else None
    base_model = AutoModelForSequenceClassification.from_pretrained(
        config.model_name,
        num_labels=2,
        low_cpu_mem_usage=True,
        torch_dtype=torch_dtype,
    )
    if hasattr(base_model, "gradient_checkpointing_enable"):
        base_model.gradient_checkpointing_enable()
    base_model.config.use_cache = False

    peft_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=config.lora_r,
        lora_alpha=config.lora_alpha,
        lora_dropout=config.lora_dropout,
        target_modules=config.lora_target_modules,
        modules_to_save=config.lora_modules_to_save,
        bias="none",
    )
    model = get_peft_model(base_model, peft_config)
    model.print_trainable_parameters()

    data_collator = DataCollatorWithPadding(
        tokenizer=tokenizer,
        pad_to_multiple_of=8 if torch.cuda.is_available() else None,
    )
    train_dataset = LazySarcasmDataset(train_df, tokenizer, input_type, config)
    validation_dataset = LazySarcasmDataset(validation_df, tokenizer, input_type, config)

    training_args = _training_args(trainer_output_dir, adapter_dir_name, config)
    trainer_kwargs = {
        "model": model,
        "args": training_args,
        "train_dataset": train_dataset,
        "eval_dataset": validation_dataset,
        "compute_metrics": trainer_compute_metrics,
        "callbacks": [EarlyStoppingCallback(early_stopping_patience=2)],
        "data_collator": data_collator,
    }
    trainer_signature = inspect.signature(Trainer.__init__)
    if "processing_class" in trainer_signature.parameters:
        trainer_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in trainer_signature.parameters:
        trainer_kwargs["tokenizer"] = tokenizer
    trainer = Trainer(**trainer_kwargs)

    checkpoint = None
    if config.resume_from_checkpoint and trainer_output_dir.exists():
        checkpoint = get_last_checkpoint(str(trainer_output_dir))
        if checkpoint:
            print(f"Resuming from checkpoint: {checkpoint}")

    start_time = time.time()
    trainer.train(resume_from_checkpoint=checkpoint)
    train_minutes = (time.time() - start_time) / 60

    model.save_pretrained(adapter_dir, safe_serialization=True)
    tokenizer.save_pretrained(adapter_dir)
    write_json(adapter_dir / "training_config.json", config.to_dict())
    print(f"Saved PEFT adapter to {adapter_dir}")

    use_windows = input_type == "context_aware" and use_windows_for_final_prediction
    if use_windows:
        diagnostics = {
            "validation": sliding_window_diagnostics(validation_df, tokenizer, config),
            "test": sliding_window_diagnostics(test_df, tokenizer, config),
        }
        write_json(out / f"{predictions_prefix}_sliding_window_diagnostics.json", diagnostics)

    validation_logits = predict_logits_dataframe(
        model,
        tokenizer,
        validation_df,
        input_type,
        config,
        use_windows=use_windows,
    )
    validation_prob = softmax(validation_logits)[:, 1]
    threshold = best_threshold_for_metric(validation_prob, validation_df["label"], metric="f1")
    validation_pred = (validation_prob >= threshold).astype(int)

    test_logits = predict_logits_dataframe(
        model,
        tokenizer,
        test_df,
        input_type,
        config,
        use_windows=use_windows,
    )
    test_prob = softmax(test_logits)[:, 1]
    test_pred = (test_prob >= threshold).astype(int)

    input_description = "comment" if input_type == "context_free" else "parent_comment + comment"
    notes = (
        f"DeBERTa-v3-base PEFT LoRA sequence classifier. "
        f"LoRA r={config.lora_r}, alpha={config.lora_alpha}, dropout={config.lora_dropout}, "
        f"target_modules={config.lora_target_modules}. Threshold={threshold:.3f} selected on validation F1. "
        f"Training time: {train_minutes:.2f} minutes."
    )
    metrics_record = compute_binary_metrics(
        y_true=test_df["label"],
        y_pred=test_pred,
        y_prob=test_prob,
        model_name=run_label,
        input_type=input_description,
        notes=notes,
        threshold=threshold,
    )
    save_metrics_record(metrics_record, metrics_filename, config)
    save_confusion_matrix(
        test_df["label"],
        test_pred,
        run_label,
        f"confusion_matrix_{predictions_prefix}.png",
        config,
    )

    validation_predictions = validation_df[["example_id", "parent_comment", "comment", "label"]].copy()
    validation_predictions = validation_predictions.rename(columns={"label": "true_label"})
    validation_predictions[f"{predictions_prefix}_pred"] = validation_pred
    validation_predictions[f"{predictions_prefix}_prob"] = validation_prob
    validation_predictions[f"{predictions_prefix}_threshold"] = threshold
    validation_predictions.to_csv(
        out / f"{predictions_prefix}_validation_predictions.csv",
        index=False,
    )

    test_predictions = test_df[["example_id", "parent_comment", "comment", "label"]].copy()
    test_predictions = test_predictions.rename(columns={"label": "true_label"})
    test_predictions[f"{predictions_prefix}_pred"] = test_pred
    test_predictions[f"{predictions_prefix}_prob"] = test_prob
    test_predictions[f"{predictions_prefix}_threshold"] = threshold
    test_predictions.to_csv(out / f"{predictions_prefix}_test_predictions.csv", index=False)

    update_comparison_outputs(config)
    del trainer, model, base_model
    cleanup_memory()
    return validation_predictions, test_predictions, metrics_record


def load_prediction_file(filename: str, config: SarcasmConfig = CONFIG) -> pd.DataFrame | None:
    path = find_file_recursive(filename, config)
    if path is None:
        return None
    print(f"Loaded predictions from {path}")
    return pd.read_csv(path, low_memory=False)


def attach_model_predictions(
    base_df: pd.DataFrame,
    predictions_df: pd.DataFrame | None,
    pred_col: str,
    prob_col: str,
) -> pd.DataFrame:
    base_df[pred_col] = np.nan
    base_df[prob_col] = np.nan
    if predictions_df is None or pred_col not in predictions_df.columns:
        return base_df
    if "example_id" in predictions_df.columns and "example_id" in base_df.columns:
        available = ["example_id", pred_col]
        if prob_col in predictions_df.columns:
            available.append(prob_col)
        small = predictions_df[available].copy()
        return base_df.drop(columns=[pred_col, prob_col]).merge(small, on="example_id", how="left")
    if len(predictions_df) == len(base_df):
        base_df[pred_col] = predictions_df[pred_col].values
        if prob_col in predictions_df.columns:
            base_df[prob_col] = predictions_df[prob_col].values
    return base_df


def create_predictions_table(test_df: pd.DataFrame, config: SarcasmConfig = CONFIG) -> pd.DataFrame:
    base = test_df[["example_id", "parent_comment", "comment", "label"]].copy()
    base = base.rename(columns={"label": "true_label"})
    prediction_specs = [
        ("tfidf_test_predictions.csv", "tfidf_pred", "tfidf_prob"),
        ("context_free_test_predictions.csv", "context_free_pred", "context_free_prob"),
        ("context_aware_test_predictions.csv", "context_aware_pred", "context_aware_prob"),
    ]
    for filename, pred_col, prob_col in prediction_specs:
        predictions = load_prediction_file(filename, config)
        if predictions is None and filename == "tfidf_test_predictions.csv":
            predictions = load_prediction_file("tfidf_predictions.csv", config)
        base = attach_model_predictions(base, predictions, pred_col, prob_col)
    base.to_csv(output_path(config) / "predictions_test.csv", index=False)
    return base


def create_error_analysis(predictions_df: pd.DataFrame, config: SarcasmConfig = CONFIG) -> pd.DataFrame:
    frame = predictions_df.copy()
    candidate_prob_cols = [
        "retrieval_augmented_prob",
        "context_aware_prob",
        "context_free_prob",
        "tfidf_prob",
    ]
    candidate_pred_cols = [
        "retrieval_augmented_pred",
        "context_aware_pred",
        "context_free_pred",
        "tfidf_pred",
    ]
    prob_col = next((col for col in candidate_prob_cols if col in frame.columns and frame[col].notna().any()), None)
    pred_col = next((col for col in candidate_pred_cols if col in frame.columns and frame[col].notna().any()), None)
    if prob_col is None or pred_col is None:
        return pd.DataFrame()
    frame["selected_prob"] = pd.to_numeric(frame[prob_col], errors="coerce")
    frame["selected_pred"] = pd.to_numeric(frame[pred_col], errors="coerce").round().astype("Int64")
    errors = frame[frame["selected_pred"] != frame["true_label"]].copy()
    errors["confidence"] = np.maximum(errors["selected_prob"], 1 - errors["selected_prob"])
    errors["error_type"] = np.where(errors["true_label"] == 1, "false_negative", "false_positive")
    errors = errors.sort_values("confidence", ascending=False)
    errors.to_csv(output_path(config) / "error_analysis.csv", index=False)
    return errors


def run_tfidf_baseline(
    train_df: pd.DataFrame,
    validation_df: pd.DataFrame,
    test_df: pd.DataFrame,
    config: SarcasmConfig = CONFIG,
) -> dict[str, Any]:
    import joblib
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import Pipeline

    start = time.time()
    tfidf_model = Pipeline(
        steps=[
            (
                "tfidf",
                TfidfVectorizer(
                    ngram_range=(1, 2),
                    max_features=100000,
                    min_df=2,
                    strip_accents="unicode",
                    lowercase=True,
                    sublinear_tf=True,
                ),
            ),
            (
                "logreg",
                LogisticRegression(
                    max_iter=1000,
                    class_weight="balanced",
                    solver="liblinear",
                    random_state=config.random_seed,
                ),
            ),
        ]
    )
    tfidf_model.fit(train_df["comment"], train_df["label"])
    minutes = (time.time() - start) / 60
    validation_prob = tfidf_model.predict_proba(validation_df["comment"])[:, 1]
    validation_pred = (validation_prob >= 0.5).astype(int)
    validation_predictions = validation_df[["example_id", "parent_comment", "comment", "label"]].copy()
    validation_predictions = validation_predictions.rename(columns={"label": "true_label"})
    validation_predictions["tfidf_pred"] = validation_pred
    validation_predictions["tfidf_prob"] = validation_prob
    validation_predictions.to_csv(output_path(config) / "tfidf_validation_predictions.csv", index=False)

    prob = tfidf_model.predict_proba(test_df["comment"])[:, 1]
    pred = (prob >= 0.5).astype(int)
    predictions = test_df[["example_id", "parent_comment", "comment", "label"]].copy()
    predictions = predictions.rename(columns={"label": "true_label"})
    predictions["tfidf_pred"] = pred
    predictions["tfidf_prob"] = prob
    predictions.to_csv(output_path(config) / "tfidf_test_predictions.csv", index=False)
    predictions.to_csv(output_path(config) / "tfidf_predictions.csv", index=False)
    joblib.dump(tfidf_model, output_path(config) / "tfidf_logreg_model.joblib")
    record = compute_binary_metrics(
        y_true=test_df["label"],
        y_pred=pred,
        y_prob=prob,
        model_name="TF-IDF + Logistic Regression",
        input_type="comment",
        notes=f"Context-free TF-IDF baseline using comment only. Training time: {minutes:.2f} minutes.",
        threshold=0.5,
    )
    save_metrics_record(record, "tfidf_metrics.json", config)
    save_confusion_matrix(test_df["label"], pred, "TF-IDF + Logistic Regression", "confusion_matrix_tfidf.png", config)
    update_comparison_outputs(config)
    return record


def build_retrieval_text(parent_comment: Any, comment: Any) -> str:
    return f"{str(parent_comment)} [REPLY] {str(comment)}"


def stratified_retrieval_sample(
    train_df: pd.DataFrame,
    validation_df: pd.DataFrame,
    test_df: pd.DataFrame,
    config: SarcasmConfig = CONFIG,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    from sklearn.model_selection import train_test_split

    if len(train_df) > config.max_retrieval_train_rows:
        retrieval_train_df, _ = train_test_split(
            train_df,
            train_size=config.max_retrieval_train_rows,
            stratify=train_df["label"],
            random_state=config.random_seed,
        )
        retrieval_train_df = retrieval_train_df.reset_index(drop=True)
    else:
        retrieval_train_df = train_df.reset_index(drop=True).copy()
    if config.max_retrieval_eval_rows is not None and len(validation_df) > config.max_retrieval_eval_rows:
        retrieval_val_df, _ = train_test_split(
            validation_df,
            train_size=config.max_retrieval_eval_rows,
            stratify=validation_df["label"],
            random_state=config.random_seed,
        )
        retrieval_val_df = retrieval_val_df.reset_index(drop=True)
    else:
        retrieval_val_df = validation_df.reset_index(drop=True).copy()
    if config.max_retrieval_eval_rows is not None and len(test_df) > config.max_retrieval_eval_rows:
        retrieval_test_df, _ = train_test_split(
            test_df,
            train_size=config.max_retrieval_eval_rows,
            stratify=test_df["label"],
            random_state=config.random_seed,
        )
        retrieval_test_df = retrieval_test_df.reset_index(drop=True)
    else:
        retrieval_test_df = test_df.reset_index(drop=True).copy()
    return retrieval_train_df, retrieval_val_df, retrieval_test_df


def build_embeddings(texts: list[str], config: SarcasmConfig = CONFIG) -> tuple[Any, str, Any]:
    try:
        from sentence_transformers import SentenceTransformer

        device = "cuda"
        try:
            import torch

            device = "cuda" if torch.cuda.is_available() else "cpu"
        except Exception:
            device = "cpu"
        model = SentenceTransformer(config.retrieval_embedding_model, device=device)
        embeddings = model.encode(
            texts,
            batch_size=config.retrieval_batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,
        )
        return np.asarray(embeddings, dtype="float32"), config.retrieval_embedding_model, model
    except Exception as exc:
        print("Sentence-transformers unavailable, falling back to TF-IDF:", exc)
        from sklearn.feature_extraction.text import TfidfVectorizer

        vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=2)
        return vectorizer.fit_transform(texts), "TF-IDF fallback", vectorizer


def encode_retrieval_queries(texts: list[str], backend: str, encoder: Any, config: SarcasmConfig = CONFIG) -> Any:
    if backend == config.retrieval_embedding_model:
        embeddings = encoder.encode(
            texts,
            batch_size=config.retrieval_batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,
        )
        return np.asarray(embeddings, dtype="float32")
    return encoder.transform(texts)


def build_retrieval_index(train_embeddings: Any, backend: str) -> tuple[Any, str]:
    if backend != "TF-IDF fallback":
        try:
            import faiss

            index = faiss.IndexFlatIP(train_embeddings.shape[1])
            index.add(train_embeddings.astype("float32"))
            return index, "FAISS cosine/IP over normalized sentence embeddings"
        except Exception as exc:
            print("FAISS unavailable, falling back to sklearn:", exc)
    from sklearn.neighbors import NearestNeighbors

    index = NearestNeighbors(metric="cosine", algorithm="auto")
    index.fit(train_embeddings)
    return index, "sklearn NearestNeighbors"


def search_retrieval_index(
    texts: list[str],
    index: Any,
    backend: str,
    encoder: Any,
    config: SarcasmConfig = CONFIG,
) -> tuple[np.ndarray, np.ndarray]:
    queries = encode_retrieval_queries(texts, backend, encoder, config)
    if index.__class__.__module__.startswith("faiss"):
        scores, indices = index.search(queries.astype("float32"), config.retrieval_k)
    else:
        distances, indices = index.kneighbors(queries, n_neighbors=config.retrieval_k)
        scores = 1 - distances
    return np.asarray(scores), np.asarray(indices)


def retrieval_features_for_frame(
    frame: pd.DataFrame,
    retrieval_train_df: pd.DataFrame,
    index: Any,
    backend: str,
    encoder: Any,
    config: SarcasmConfig = CONFIG,
) -> pd.DataFrame:
    texts = [
        build_retrieval_text(parent, comment)
        for parent, comment in zip(frame["parent_comment"], frame["comment"])
    ]
    all_scores = []
    all_indices = []
    for start in range(0, len(texts), config.retrieval_batch_size * 8):
        batch = texts[start : start + config.retrieval_batch_size * 8]
        scores, indices = search_retrieval_index(batch, index, backend, encoder, config)
        all_scores.append(scores)
        all_indices.append(indices)
    scores = np.vstack(all_scores)
    indices = np.vstack(all_indices)
    neighbor_labels = retrieval_train_df.iloc[indices.reshape(-1)]["label"].to_numpy().reshape(indices.shape)
    weights = np.maximum(scores, 0.0)
    weight_sums = weights.sum(axis=1)
    unweighted_prob = neighbor_labels.mean(axis=1)
    weighted_prob = np.divide(
        (neighbor_labels * weights).sum(axis=1),
        weight_sums,
        out=unweighted_prob.astype(float).copy(),
        where=weight_sums > 0,
    )
    features = frame[["example_id", "label"]].rename(columns={"label": "true_label"}).copy()
    features["retrieval_knn_prob"] = weighted_prob
    features["retrieval_knn_pred"] = (weighted_prob >= 0.5).astype(int)
    features["retrieval_top1_label"] = neighbor_labels[:, 0]
    features["retrieval_top1_score"] = scores[:, 0]
    features["retrieval_score_mean"] = scores.mean(axis=1)
    features["retrieval_score_std"] = scores.std(axis=1)
    features["retrieval_score_gap"] = scores[:, 0] - scores[:, 1]
    features["retrieval_positive_neighbor_count"] = neighbor_labels.sum(axis=1)
    return features


def merge_supervised_probs(features: pd.DataFrame, split: str, config: SarcasmConfig = CONFIG) -> pd.DataFrame:
    tfidf_filename = "tfidf_validation_predictions.csv" if split == "validation" else "tfidf_test_predictions.csv"
    candidates = [
        (tfidf_filename, "tfidf_prob"),
        (f"context_free_{split}_predictions.csv", "context_free_prob"),
        (f"context_aware_{split}_predictions.csv", "context_aware_prob"),
    ]
    merged = features.copy()
    for filename, column in candidates:
        predictions = load_prediction_file(filename, config)
        if predictions is None and filename == "tfidf_test_predictions.csv":
            predictions = load_prediction_file("tfidf_predictions.csv", config)
        if predictions is None or column not in predictions.columns:
            continue
        merged = merged.merge(predictions[["example_id", column]], on="example_id", how="left")
    return merged


def train_validation_calibrated_stacker(
    validation_features: pd.DataFrame,
    test_features: pd.DataFrame,
    config: SarcasmConfig = CONFIG,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    feature_columns = [
        column
        for column in [
            "tfidf_prob",
            "context_free_prob",
            "context_aware_prob",
            "retrieval_knn_prob",
            "retrieval_top1_label",
            "retrieval_top1_score",
            "retrieval_score_mean",
            "retrieval_score_std",
            "retrieval_score_gap",
            "retrieval_positive_neighbor_count",
        ]
        if column in validation_features.columns and validation_features[column].notna().any()
    ]
    if "context_aware_prob" not in feature_columns:
        raise ValueError(
            "Validation-calibrated stacker requires context_aware validation predictions. "
            "Run Notebook 3 before Notebook 4."
        )
    x_val = validation_features[feature_columns].astype(float).to_numpy()
    x_test = test_features[feature_columns].astype(float).to_numpy()
    x_val = np.nan_to_num(x_val, nan=0.5, posinf=1.0, neginf=0.0)
    x_test = np.nan_to_num(x_test, nan=0.5, posinf=1.0, neginf=0.0)
    y_val = validation_features["true_label"].astype(int).to_numpy()
    y_test = test_features["true_label"].astype(int).to_numpy()

    stacker = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, class_weight="balanced", random_state=config.random_seed),
    )
    stacker.fit(x_val, y_val)
    val_prob = stacker.predict_proba(x_val)[:, 1]
    threshold = best_threshold_for_metric(val_prob, y_val, metric="f1")
    test_prob = stacker.predict_proba(x_test)[:, 1]
    test_pred = (test_prob >= threshold).astype(int)
    output = test_features.copy()
    output["retrieval_augmented_prob"] = test_prob
    output["retrieval_augmented_pred"] = test_pred
    output["retrieval_augmented_threshold"] = threshold
    record = compute_binary_metrics(
        y_true=y_test,
        y_pred=test_pred,
        y_prob=test_prob,
        model_name="Retrieval-augmented validation-calibrated stacker",
        input_type="validation-trained supervised probabilities + retrieval-neighbor features",
        notes=(
            f"Stacker trained and thresholded on validation only using features: {', '.join(feature_columns)}. "
            "The test split is untouched until final evaluation."
        ),
        threshold=threshold,
    )
    output.to_csv(output_path(config) / "retrieval_augmented_predictions.csv", index=False)
    save_metrics_record(record, "retrieval_metrics.json", config)
    update_comparison_outputs(config)
    return output, record


## Configuration

`debug=False` and `max_transformer_*_rows=None` ensure all three models train on the **full dataset**.  
Set `debug=True` to subsample 20 k rows for a fast CPU smoke test.


In [ ]:
from IPython.display import display
import textwrap

config = cfg(
    debug=False,                        # set True for a 20 k-row CPU smoke test
    max_transformer_train_rows=None,    # None = entire training split (GPU)
    max_transformer_eval_rows=None,     # None = entire val/test splits
    run_tfidf=True,
    run_context_free_transformer=True,
    run_context_aware_transformer=True,
    run_rag=True,
    run_llm_prompts=True,
)

set_all_seeds(config.random_seed, deterministic=False)
save_run_metadata(config)
prepare_input_search(config)

print('Output directory :', output_path(config))
print('Model            :', config.model_name)
print('LoRA r / alpha   :', config.lora_r, '/', config.lora_alpha)
print('debug            :', config.debug)
display(version_log(config)['gpus'])


## Step 1 — Load, clean and split the dataset


In [ ]:
df = load_and_clean_dataset(config)
display(df.head())
display(df['label'].value_counts().rename('count').to_frame())


In [ ]:
train_df, validation_df, test_df = create_splits(df, config)
save_split_artifacts(train_df, validation_df, test_df, config)

diagnostics = save_length_diagnostics(train_df, validation_df, test_df, config)
summary = []
for split_name, split_diag in diagnostics.items():
    row = {'split': split_name}
    row.update({f'pair_words_{k}': v for k, v in split_diag['parent_plus_comment_words'].items()})
    row['pct_pair_words_gt_256'] = split_diag['rows_above_pair_word_threshold']['256']['pct']
    summary.append(row)
display(pd.DataFrame(summary))


## Step 2 — TF-IDF + Logistic Regression baseline

Trains on the **full training split** using reply text only.  
100 k features, unigrams + bigrams, balanced class weights.


In [ ]:
if config.run_tfidf:
    tfidf_metrics = run_tfidf_baseline(train_df, validation_df, test_df, config)
    display(pd.DataFrame([tfidf_metrics])[COMPARISON_COLUMNS])
else:
    print('TF-IDF skipped.')


## Steps 3 & 4 — DeBERTa-v3-base + LoRA fine-tuning

Both transformer models are trained with **PEFT LoRA** on the same split.  
LoRA configuration:
- `r = 16`, `lora_alpha = 32`, `lora_dropout = 0.05`
- Target modules: `query_proj`, `key_proj`, `value_proj`
- Modules to save (full weights): `classifier`, `pooler`

Training setup: FP16/BF16 mixed precision, gradient checkpointing, gradient accumulation (4 steps), early stopping (patience=2), AdamW, lr=2e-4.


In [ ]:
train_t, val_t, test_t = prepare_transformer_runtime_frames(
    train_df, validation_df, test_df, config
)
print('Transformer splits:', train_t.shape, val_t.shape, test_t.shape)


### Step 3 — Context-free DeBERTa LoRA

Input: reply comment text only.  
Adapter saved to `deberta_v3_base_context_free_lora_adapter/`.


In [ ]:
if config.run_context_free_transformer:
    cf_val_preds, cf_test_preds, cf_metrics = train_deberta_lora_classifier(
        train_df=train_t,
        validation_df=val_t,
        test_df=test_t,
        input_type='context_free',
        run_label='Context-free DeBERTa-v3-base LoRA',
        adapter_dir_name='deberta_v3_base_context_free_lora_adapter',
        predictions_prefix='context_free',
        metrics_filename='context_free_metrics.json',
        config=config,
        use_windows_for_final_prediction=False,
    )
    display(pd.DataFrame([cf_metrics])[COMPARISON_COLUMNS])
else:
    print('Context-free transformer skipped.')


### Step 4 — Context-aware DeBERTa LoRA  ← primary model

Input: `parent_comment + comment` as a sentence pair.  
For examples exceeding 256 tokens, up to 4 sliding windows of the parent context (stride=64) are used; their logits are averaged.  
Adapter saved to `deberta_v3_base_context_aware_lora_adapter/`.


In [ ]:
if config.run_context_aware_transformer:
    ca_val_preds, ca_test_preds, ca_metrics = train_deberta_lora_classifier(
        train_df=train_t,
        validation_df=val_t,
        test_df=test_t,
        input_type='context_aware',
        run_label='Context-aware DeBERTa-v3-base LoRA',
        adapter_dir_name='deberta_v3_base_context_aware_lora_adapter',
        predictions_prefix='context_aware',
        metrics_filename='context_aware_metrics.json',
        config=config,
        use_windows_for_final_prediction=True,
    )
    display(pd.DataFrame([ca_metrics])[COMPARISON_COLUMNS])
else:
    print('Context-aware transformer skipped.')

predictions_test = create_predictions_table(test_df, config)
error_analysis   = create_error_analysis(predictions_test, config)
display(predictions_test.head())
display(error_analysis.head(10))


## Step 5 — Retrieval-augmented ensemble

Embeds up to 50 k training examples with `BAAI/bge-base-en-v1.5`, indexes them with FAISS (cosine/IP), retrieves k=5 neighbours per query, then trains a logistic stacker **on validation features only** combining all supervised model probabilities with k-NN similarity features.


In [ ]:
if config.run_rag:
    retrieval_train_df, retrieval_validation_df, retrieval_test_df = \
        stratified_retrieval_sample(train_df, validation_df, test_df, config)
    train_texts = [
        build_retrieval_text(p, c)
        for p, c in zip(retrieval_train_df['parent_comment'], retrieval_train_df['comment'])
    ]
    train_embeddings, embedding_backend, retrieval_encoder = build_embeddings(train_texts, config)
    retrieval_index, index_backend = build_retrieval_index(train_embeddings, embedding_backend)
    print('Embedding backend:', embedding_backend)
    print('Index backend    :', index_backend)
    print('Retrieval splits :', retrieval_train_df.shape, retrieval_validation_df.shape, retrieval_test_df.shape)
else:
    print('RAG skipped.')


In [ ]:
if config.run_rag:
    validation_features = retrieval_features_for_frame(
        retrieval_validation_df, retrieval_train_df, retrieval_index,
        embedding_backend, retrieval_encoder, config,
    )
    test_features = retrieval_features_for_frame(
        retrieval_test_df, retrieval_train_df, retrieval_index,
        embedding_backend, retrieval_encoder, config,
    )
    validation_features = merge_supervised_probs(validation_features, 'validation', config)
    test_features       = merge_supervised_probs(test_features,       'test',       config)
    test_features = test_features.merge(
        test_df[['example_id', 'parent_comment', 'comment']], on='example_id', how='left'
    )
    write_table(validation_features, 'retrieval_validation_features', config)
    write_table(test_features,       'retrieval_test_features',       config)

    retrieval_knn_metrics = compute_binary_metrics(
        y_true=test_features['true_label'],
        y_pred=test_features['retrieval_knn_pred'],
        y_prob=test_features['retrieval_knn_prob'],
        model_name=f'Retrieval-only kNN diagnostic (k={config.retrieval_k})',
        input_type='parent_comment + comment + retrieved examples',
        notes=f'Diagnostic: {embedding_backend} + {index_backend}.',
        threshold=0.5,
    )
    write_json(output_path(config) / 'retrieval_knn_diagnostic_metrics.json', retrieval_knn_metrics)
    display(pd.DataFrame([retrieval_knn_metrics])[COMPARISON_COLUMNS])


In [ ]:
if config.run_rag:
    retrieval_augmented_predictions, retrieval_metrics = train_validation_calibrated_stacker(
        validation_features, test_features, config,
    )
    display(pd.DataFrame([retrieval_metrics])[COMPARISON_COLUMNS])
    error_analysis = create_error_analysis(retrieval_augmented_predictions, config)
    display(error_analysis.head(10))


## Step 6 — LLM prompt artifacts

Generates zero-shot and retrieval-augmented few-shot prompt templates for test examples. No paid API call is made — the prompts are saved as CSVs and a text file for offline or API-based evaluation.


In [ ]:
def build_zero_shot_prompt(parent_comment, comment):
    return textwrap.dedent(f"""
        You are judging sarcasm in a Reddit conversation.

        Parent comment:
        {parent_comment}

        Target reply:
        {comment}

        Is the target reply sarcastic? Answer 0 or 1 and explain briefly.
        Use 1 for sarcastic and 0 for non-sarcastic.
    """).strip()


def retrieve_similar_examples(parent_comment, comment, k=5):
    if not config.run_rag:
        return pd.DataFrame()
    texts = [build_retrieval_text(parent_comment, comment)]
    scores, indices = search_retrieval_index(
        texts, retrieval_index, embedding_backend, retrieval_encoder, config,
    )
    rows = retrieval_train_df.iloc[indices[0]][['parent_comment', 'comment', 'label']].copy()
    rows['similarity_score'] = scores[0]
    return rows.reset_index(drop=True)


def build_few_shot_prompt(parent_comment, comment, retrieved_examples):
    blocks = []
    for i, (_, ex) in enumerate(retrieved_examples.head(3).iterrows(), start=1):
        blocks.append(textwrap.dedent(f"""
            Example {i}
            Parent comment: {ex['parent_comment']}
            Target reply: {ex['comment']}
            Label: {int(ex['label'])}
        """).strip())
    examples_text = '\n\n'.join(blocks) if blocks else 'No retrieved examples available.'
    return textwrap.dedent(f"""
        You are judging sarcasm in a Reddit conversation.

        Here are similar labeled examples:

        {examples_text}

        Now classify this new case.

        Parent comment:
        {parent_comment}

        Target reply:
        {comment}

        Is the target reply sarcastic? Answer 0 or 1 and explain briefly.
        Use 1 for sarcastic and 0 for non-sarcastic.
    """).strip()


In [ ]:
prompt_rows = []
prompt_text_blocks = []

if config.run_llm_prompts:
    prompt_examples = predictions_test.copy()
    if len(prompt_examples) > config.max_llm_prompt_rows:
        from sklearn.model_selection import train_test_split as _tts
        prompt_examples, _ = _tts(
            prompt_examples,
            train_size=config.max_llm_prompt_rows,
            stratify=prompt_examples['true_label'],
            random_state=config.random_seed,
        )
    prompt_examples = prompt_examples.reset_index(drop=True)

    for qn, (_, row) in enumerate(prompt_examples.iterrows(), start=1):
        retrieved = retrieve_similar_examples(row['parent_comment'], row['comment'], k=config.retrieval_k)
        z_prompt = build_zero_shot_prompt(row['parent_comment'], row['comment'])
        f_prompt = build_few_shot_prompt(row['parent_comment'], row['comment'], retrieved)
        prompt_rows.append({
            'query_number': qn,
            'parent_comment': row['parent_comment'],
            'comment': row['comment'],
            'true_label': row['true_label'],
            'zero_shot_prompt': z_prompt,
            'few_shot_prompt': f_prompt,
        })
        if qn <= 5:
            prompt_text_blocks.append(
                f'=== Query {qn}: zero-shot ===\n' + z_prompt +
                f'\n\n=== Query {qn}: few-shot ===\n' + f_prompt
            )

llm_prompt_df = pd.DataFrame(prompt_rows)
llm_template_df = llm_prompt_df[['query_number', 'parent_comment', 'comment', 'true_label']].copy() \
    if len(llm_prompt_df) else pd.DataFrame(
        columns=['query_number', 'parent_comment', 'comment', 'true_label']
    )
for col in ['zero_shot_pred', 'zero_shot_prob', 'few_shot_pred', 'few_shot_prob']:
    llm_template_df[col] = pd.NA

llm_prompt_df.to_csv(output_path(config) / 'llm_prompt_examples.csv', index=False)
llm_prompt_df.to_csv(output_path(config) / 'llm_prompt_eval.csv', index=False)
llm_template_df.to_csv(output_path(config) / 'llm_predictions_template.csv', index=False)
(output_path(config) / 'llm_prompt_examples.txt').write_text('\n\n'.join(prompt_text_blocks), encoding='utf-8')
display(llm_prompt_df.head(2))


In [ ]:
# To evaluate LLM predictions: fill llm_predictions_template.csv and place it in the working directory.
llm_predictions_path = find_file_recursive('llm_predictions.csv', config)
llm_metrics_list = []
if llm_predictions_path is not None:
    llm_predictions = pd.read_csv(llm_predictions_path)
    for model_name, pred_col, prob_col in [
        ('LLM zero-shot classifier', 'zero_shot_pred', 'zero_shot_prob'),
        ('LLM few-shot retrieval-augmented classifier', 'few_shot_pred', 'few_shot_prob'),
    ]:
        if pred_col in llm_predictions.columns:
            rec = compute_binary_metrics(
                y_true=llm_predictions['true_label'],
                y_pred=llm_predictions[pred_col],
                y_prob=llm_predictions[prob_col] if prob_col in llm_predictions.columns else None,
                model_name=model_name,
                input_type='LLM prompt',
                notes=f'Evaluated from {llm_predictions_path.name}.',
            )
            llm_metrics_list.append(rec)
    if llm_metrics_list:
        write_json(output_path(config) / 'llm_metrics.json', llm_metrics_list)
        update_comparison_outputs(config)
        display(pd.DataFrame(llm_metrics_list)[COMPARISON_COLUMNS])
else:
    print('No llm_predictions.csv found — prompt files are qualitative artifacts only.')


## Final metrics comparison across all models


In [ ]:
metrics_comparison = update_comparison_outputs(config)
display(
    metrics_comparison[COMPARISON_COLUMNS]
    .sort_values('f1', ascending=False)
    .reset_index(drop=True)
)


## Save all outputs


In [ ]:
all_outputs = [
    'run_config.json', 'version_log.json', 'length_diagnostics.json',
    'train_split.csv', 'validation_split.csv', 'test_split.csv',
    'train_split.parquet', 'validation_split.parquet', 'test_split.parquet',
    'tfidf_metrics.json', 'tfidf_test_predictions.csv',
    'tfidf_validation_predictions.csv', 'tfidf_logreg_model.joblib',
    'confusion_matrix_tfidf.png',
    'deberta_v3_base_context_free_lora_adapter',
    'context_free_metrics.json',
    'context_free_test_predictions.csv', 'context_free_validation_predictions.csv',
    'confusion_matrix_context_free.png',
    'deberta_v3_base_context_aware_lora_adapter',
    'context_aware_metrics.json',
    'context_aware_test_predictions.csv', 'context_aware_validation_predictions.csv',
    'context_aware_sliding_window_diagnostics.json',
    'confusion_matrix_context_aware.png',
    'predictions_test.csv', 'error_analysis.csv',
    'retrieval_validation_features.csv', 'retrieval_test_features.csv',
    'retrieval_augmented_predictions.csv',
    'retrieval_metrics.json', 'retrieval_knn_diagnostic_metrics.json',
    'llm_prompt_examples.csv', 'llm_prompt_eval.csv',
    'llm_predictions_template.csv', 'llm_prompt_examples.txt',
    'metrics_comparison.csv',
]
zip_outputs('sarcasm_unified_outputs.zip', all_outputs, config)
print('Pipeline complete.')
